# 04 — LightGBM Training
## Dynamic Pricing & Demand Forecasting
### Goal: Train LightGBM model to beat baseline RMSE of 2.0970

## Section 1: Setup & Data Loading

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import pickle
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

print("Imports successful!")
print(f"LightGBM version: {lgb.__version__}")

### 1.2 Load data & split

In [ ]:
# Load processed data
df = pd.read_pickle('../data/processed/df_train.pkl')

# Target and features
TARGET = 'sales'
EXCLUDE = ['id', 'sales', 'date', 'wm_yr_wk']
FEATURES = [c for c in df.columns if c not in EXCLUDE]

# Time based split
cutoff_date = df['date'].max() - pd.Timedelta(days=28)

train = df[df['date'] <= cutoff_date]
val   = df[df['date'] >  cutoff_date]

# Feature matrices
X_train = train[FEATURES]
y_train = train[TARGET]
X_val   = val[FEATURES]
y_val   = val[TARGET]

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"Features: {len(FEATURES)}")
print(f"\nBaseline RMSE to beat: 2.2186")

## Section 2: LightGBM Training
### 2.1 Initial LightGBM model
Start with reasonable params — tune later with Optuna

In [ ]:
# LightGBM datasets
dtrain = lgb.Dataset(
    X_train, 
    label=y_train,
    categorical_feature=['item_id', 'dept_id', 'cat_id', 
                         'store_id', 'state_id'],
    free_raw_data=False
)

dval = lgb.Dataset(
    X_val,
    label=y_val,
    reference=dtrain,
    free_raw_data=False
)

print("LightGBM datasets created!")
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")

In [ ]:
# LightGBM datasets
dtrain = lgb.Dataset(
    X_train, 
    label=y_train,
    categorical_feature=['item_id', 'dept_id', 'cat_id', 
                         'store_id', 'state_id'],
    free_raw_data=False
)

dval = lgb.Dataset(
    X_val,
    label=y_val,
    reference=dtrain,
    free_raw_data=False
)

print("LightGBM datasets created!")
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")

### 2.2 Initial params

In [ ]:
params = {
    'objective':        'tweedie',      # best for count/retail data
    'tweedie_variance_power': 1.1,      # close to Poisson
    'metric':           'rmse',
    'learning_rate':    0.05,
    'num_leaves':       128,
    'min_child_samples': 20,
    'subsample':        0.8,
    'subsample_freq':   1,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'n_jobs':           -1,
    'seed':             42,
    'verbose':          -1
}

print("Parameters set:")
for k, v in params.items():
    print(f"  {k}: {v}")

### 2.3 Train model

In [ ]:
# Train with early stopping
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50)
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=callbacks
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best val RMSE:  {model.best_score['val']['rmse']:.4f}")

In [ ]:
# Section 3: Optuna Hyperparameter Tuning
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        'objective':              'tweedie',
        'tweedie_variance_power': 1.1,
        'metric':                 'rmse',
        'verbosity':              -1,
        'boosting_type':          'gbdt',
        'n_jobs':                 -1,
        'seed':                   42,
        
        # parameters to tune
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 64, 512),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq':     1,
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':         trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    
    callbacks = [
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=-1)
    ]
    
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        valid_names=['val'],
        callbacks=callbacks
    )
    
    return model.best_score['val']['rmse']

# run optimisation
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest RMSE:   {study.best_value:.4f}")
print(f"Best params: {study.best_params}")